### Ingestão — Megawhat (megawhat.uol.com.br)

- **Feed:** `https://megawhat.uol.com.br/destaques-do-diario/feed`.
- **Escopo:** site de nicho único (energia), como o CreditoPrivado360 — feed geral, sem filtro de categoria.
- **Download:** confirmado manualmente — corpo do artigo vem completo, sem depender de JavaScript, sem paywall.



In [0]:
%pip install --quiet feedparser beautifulsoup4 httpx lxml curl_cffi
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from typing import Optional

import feedparser
import httpx
from bs4 import BeautifulSoup

# fallback nesse caso
from curl_cffi import requests as cffi_requests


/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/ENERGIA"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Sem filtro de categoria: site de nicho único (energia).

FEEDS_A_COMBINAR = [
    "https://megawhat.uol.com.br/destaques-do-diario/feed",
    "https://megawhat.uol.com.br/ultimas-noticias/feed",
]

JANELA_HORAS = 24

SOURCE_ID = "megawhat_destaques"
SOURCE_DESCRICAO = "Linked from Megawhat — Destaques do Diário"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]

HTTP_TIMEOUT = 30

MIN_CHARS_TEXTO = 200


[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-27


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def parsear_data_rss(data_str: str) -> str:
    try:
        return parsedate_to_datetime(data_str).strftime("%Y-%m-%d")
    except Exception:
        return HOJE


def parece_paywall(texto: str) -> bool:
    marcadores = [
        "para continuar lendo",
        "assine agora mesmo",
        "assine já",
        "conteúdo exclusivo para assinantes",
        "faça login para ler",
        "este conteúdo é para assinantes",
        "cadastre-se para continuar",
        "conteúdo para assinantes uol",
    ]
    trecho = texto[:1500].lower()
    return any(m in trecho for m in marcadores)


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    ua = random.choice(USER_AGENTS)
    headers = {
        "User-Agent": ua,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
                  "image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
# =============================================================================
# Etapa 1 — Verificação de feed
# =============================================================================

def buscar_itens_de_feed(url: str, tentativas: int = 2) -> list:
    """Busca um feed específico, com retry. Retorna lista de entries (vazia se falhar)."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, timeout=15, follow_redirects=True,
                              headers={"User-Agent": random.choice(USER_AGENTS)})
        except Exception as e:
            print(f"[feed] {url} -> tentativa {tentativa}/{tentativas} falhou: {e}")
            if tentativa < tentativas:
                time.sleep(random.uniform(1.0, 2.5))
            continue

        if resp.status_code != 200:
            print(f"[feed] {url} -> HTTP {resp.status_code}.")
            return []

        parsed = feedparser.parse(resp.content)
        print(f"[feed] OK: '{url}', {len(parsed.entries)} entries.")
        return parsed.entries

    print(f"[feed] {url} -> todas as tentativas falharam, seguindo sem esse feed.")
    return []


def combinar_feeds(urls: list) -> list:
    """Busca todos os feeds da lista e junta os itens, removendo duplicata por URL."""
    vistos = set()
    combinados = []

    for url in urls:
        for entry in buscar_itens_de_feed(url):
            link = entry.get("link")
            if link and link in vistos:
                continue
            if link:
                vistos.add(link)
            combinados.append(entry)

    print(f"[feed] Total combinado: {len(combinados)} itens únicos de {len(urls)} feeds.")
    return combinados

In [0]:
# =============================================================================
# Etapa 2 — Verificação de janela de tempo
# =============================================================================
# Sem filtro de categoria (site de nicho único).

def dentro_da_janela(entry, horas: int = JANELA_HORAS) -> bool:
    if not entry.get("published_parsed"):
        print(f"    -> aviso: item sem published_parsed ({entry.get('title', '?')[:60]}); incluindo mesmo assim.")
        return True

    pub_dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
    limite = datetime.now(timezone.utc) - timedelta(hours=horas)
    return pub_dt >= limite


def selecionar_itens(itens: list) -> list:
    antes = len(itens)
    itens = [e for e in itens if dentro_da_janela(e)]
    print(f"[filtro] janela de {JANELA_HORAS}h: {antes} itens -> {len(itens)} dentro do prazo.")
    return itens


In [0]:
# =============================================================================
# Etapa 3 — Download do HTML da matéria
# =============================================================================

def baixar_html(url: str) -> Optional[str]:
    headers = headers_aleatorios(referer="https://megawhat.uol.com.br/")

    time.sleep(random.uniform(0.5, 1.5))

    try:
        resp = httpx.get(url, headers=headers, timeout=20, follow_redirects=True)
        if resp.status_code == 200 and resp.text:
            return resp.text
        print(f"    -> httpx retornou HTTP {resp.status_code}; tentando fallback curl_cffi.")
    except Exception as e:
        print(f"    -> httpx falhou ({e}); tentando fallback curl_cffi.")

    try:
        resp = cffi_requests.get(url, headers=headers, impersonate=random.choice(IMPERSONATE_PROFILES), timeout=20)
        if resp.status_code == 200 and resp.text:
            return resp.text
        print(f"    -> curl_cffi também retornou HTTP {resp.status_code}.")
    except Exception as e:
        print(f"    -> curl_cffi também falhou: {e}")

    return None


In [0]:
# =============================================================================
# Etapa 4 — Limpar o HTML e extrair só o texto útil
# =============================================================================

TAGS_LIXO = [
    "script", "style", "noscript", "iframe", "svg", "form",
    "nav", "footer", "header", "aside", "button",
]

def extrair_texto(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    article = soup.find("article")
    base = article if article else soup

    texto = base.get_text("\n", strip=True)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


In [0]:
# =============================================================================
# Etapa 5 — Salvar no Volume
# =============================================================================

def salvar_artefatos(
    pasta: str,
    source: str,
    titulo: str,
    texto: str,
    metadados: dict,
) -> tuple[str, str]:
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 6 — Pipeline principal
# =============================================================================

def processar_entry(entry) -> Optional[dict]:
    titulo = entry.get("title", "sem-titulo")
    url = entry.get("link")
    print(f"\n  [item] {titulo[:100]}")

    if not url:
        print("    -> sem link; pulando.")
        return None

    html = baixar_html(url)
    if not html:
        print("    -> download do HTML falhou; pulando.")
        return None

    texto = extrair_texto(html)

    if parece_paywall(texto):
        print("    -> marcador de paywall detectado; pulando.")
        return None

    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    data_publicacao = parsear_data_rss(entry.get("published", ""))

    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(
        pasta=PASTA_DESTINO,
        source=SOURCE_ID,
        titulo=titulo,
        texto=texto,
        metadados=metadados,
    )
    print(f"    -> salvo em {caminho_txt}")

    return {
        "titulo": titulo,
        "url": url,
        "caminho_txt": caminho_txt,
        "caminho_json": caminho_json,
    }


In [0]:
# =============================================================================
# Execução
# =============================================================================

itens_brutos = combinar_feeds(FEEDS_A_COMBINAR)
itens = selecionar_itens(itens_brutos)

print(f"\n=== {len(itens)} itens a processar (combinados de {len(FEEDS_A_COMBINAR)} feeds) ===")

todos_resultados: list[dict] = []
for entry in itens:
    try:
        resultado = processar_entry(entry)
        if resultado:
            todos_resultados.append(resultado)
    except Exception as e:
        print(f"[ERRO] item {entry.get('title', '?')!r} falhou: {e}")

print(f"\n\n=== Fim. {len(todos_resultados)} matérias salvas em {PASTA_DESTINO} ===")
print(f"=== Feeds combinados nesta execução: {', '.join(FEEDS_A_COMBINAR)} ===")


[feed] OK: 'https://megawhat.uol.com.br/destaques-do-diario/feed', 10 entries.
[feed] OK: 'https://megawhat.uol.com.br/ultimas-noticias/feed', 10 entries.
[feed] Total combinado: 18 itens únicos de 2 feeds.
[filtro] janela de 24h: 18 itens -> 0 dentro do prazo.

=== 0 itens a processar (combinados de 2 feeds) ===


=== Fim. 0 matérias salvas em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-27 ===
=== Feeds combinados nesta execução: https://megawhat.uol.com.br/destaques-do-diario/feed, https://megawhat.uol.com.br/ultimas-noticias/feed ===
